In [1]:
from langchain_community.llms.ollama import Ollama
import re
from langchain.vectorstores import FAISS
import pandas as pd
from build_database import get_embedding_function

llama_model = Ollama(model="llama3.2")
mistral_model = Ollama(model ='mistral')

/var/folders/p3/jjp2dcm14lvfdvj54q_s4g080000gn/T/ipykernel_12417/2710079494.py:7: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaLLM``.
  llama_model = Ollama(model="llama3.2")


In [10]:
# Hyperparameter Tuning Wrapper
class ModelHyperparameterTuning:
    def __init__(self, chunk_size, method):
        self.chunk_size = chunk_size
        self.method = method

    def load_vignettes(self, filepath, cat):
        """Load vignettes, categories, and ground truth labels from the CSV file."""
        data = pd.read_csv(filepath)
        stress_vignettes = data[data["Category"] == cat]
        vignettes = {}
        for _, row in stress_vignettes.iterrows():
            vignette_id = row['Vignette ID']
            category = row['Category']
            vignette = f"{row['Referral']}\n{row['Presenting Symptoms']}\n{row['Additional Background Information']}"
            vignettes[vignette_id] = {
                "category": category,
                "vignette_text": vignette.strip(),
                "label": row['Label']
            }
        return vignettes

    def generate_diagnosis(self, rag_text, vignette_text, cot_text):
        prompt = f"""
        You are a professional clinician tasked with diagnosing based on the following vignette. 
        Consider clinical guidelines carefully to reach a thoughtful diagnosis based on the vignette's details.

        If you don't know the answer, just say that you don't know; don't try to make up an answer.

        Clinical Guidelines:
        {rag_text}

        Vignette:
        {vignette_text}
        """
        response = llama_model.invoke(prompt)
        return response

    def rag(self, vignette):
        embedding_function = get_embedding_function()
        db = FAISS.load_local("Chroma_path", embedding_function, allow_dangerous_deserialization=True)
        results = db.similarity_search(vignette, k=self.chunk_size)
        context_text = "\n\n---\n\n".join([doc.page_content for doc in results])
        return context_text

    def rag_compress(self, vignette):
        context_text = self.rag(vignette)
        prompt = (
            "You are a professional summarizer and content organizer. Given the following extracted "
            "information, compress and restructure it into a concise, well-organized summary while "
            "preserving all critical details. Ensure the language is clear and the structure is logical.\n\n"
            "Extracted Chunks:\n"
            f"{context_text}\n\n"
            "Compressed and Restructured Summary:"
        )
        compress_response = llama_model.invoke(prompt)
        return compress_response

    def cot(self): #cot file name need to update
        with open("cot_stress.txt", "r", encoding="utf-8") as file:
            cot_text = file.read()
        return cot_text

    def cot_compress(self):
        cot_text = self.cot()
        prompt = (
            "You are a professional summarizer and content organizer. Given the following extracted "
            "information, compress and restructure it into a concise, well-organized summary while "
            "preserving all critical details. Ensure the language is clear and the structure is logical.\n\n"
            "Extracted Chunks:\n"
            f"{cot_text}\n\n"
            "Compressed and Restructured Summary:"
        )
        compress_response = llama_model.invoke(prompt)
        return compress_response

    def get_rag_text(self, vignette_text):
        if self.method in ["rag only", "rag compress"]:
            return self.rag_compress(vignette_text) if "compress" in self.method else self.rag(vignette_text)
        return ""

    def get_cot_text(self):
        if self.method in ["cot only", "cot compress"]:
            return self.cot_compress() if "compress" in self.method else self.cot()
        return ""

    def extract_diagnosis(self, vignette_text):
        """Use the model to extract the diagnosis from the vignette text."""
        prompt = f"Extract the final diagnosis from this text:\n\n{vignette_text}"
        response = llama_model(prompt)
        return response.strip()

    def compare_diagnosis(self, model_diagnosis, ground_truth_label):
        """Use the model to determine if the extracted diagnosis matches the ground truth."""
        prompt = (
            f"Given the diagnosis: '{model_diagnosis}'\n\n"
            f"The ground truth diagnosis is: '{ground_truth_label}'.\n\n"
            f"Does the final diagnosis match the ground truth? Give the similarity score (0-100) and provide reasoning."
        )
        response = mistral_model(prompt).strip()
        score_match = re.search(r"(\d+)", response)
        if score_match:
            score = int(score_match.group(1))
        else:
            raise ValueError("Unable to parse similarity score from the response.")
        return score

    def evaluate_model(self, vignettes):
        """Evaluate model performance by comparing predictions with ground truth labels."""
        correct_counts = {"Mood": 0, "Anxiety": 0, "Stress": 0}
        total_counts = {"Mood": 0, "Anxiety": 0, "Stress": 0}
        overall_correct = 0
        total_count = len(vignettes)

        cot_text = self.get_cot_text()

        for vignette_id, vignette_data in vignettes.items():
            category = vignette_data['category']
            vignette_text = vignette_data['vignette_text']
            ground_truth_label = vignette_data['label']

            rag_text = self.get_rag_text(vignette_text)
            model_diagnosis = self.generate_diagnosis(rag_text, vignette_text, cot_text).strip()
            final_diag = self.extract_diagnosis(model_diagnosis)

            score = self.compare_diagnosis(final_diag, ground_truth_label)
            if score >= 80:
                correct_counts[category] += 1
                overall_correct += 1
            total_counts[category] += 1

            print("------------------------------------------------------------------------------")
            print(f"Vignette ID: {vignette_id}")
            print(f"Category: {category}")
            print(f"Vignette Text:\n{vignette_text}\n")
            print(f"Model Diagnosis: {model_diagnosis}")
            print(f"The extracted Model Diagnosis: {final_diag}")
            print(f"Ground Truth Label: {ground_truth_label}")
            print("The similarity score provided by the model:", score)

        for category in correct_counts:
            category_accuracy = correct_counts[category] / total_counts[category] if total_counts[category] > 0 else 0
            print(f"Accuracy for {category}: {category_accuracy:.2f}")

        overall_accuracy = overall_correct / total_count if total_count > 0 else 0
        print(f"Overall Accuracy: {overall_accuracy:.2f}")

        return {
            "Mood": correct_counts["Mood"] / total_counts["Mood"] if total_counts["Mood"] > 0 else 0,
            "Anxiety": correct_counts["Anxiety"] / total_counts["Anxiety"] if total_counts["Anxiety"] > 0 else 0,
            "Stress": correct_counts["Stress"] / total_counts["Stress"] if total_counts["Stress"] > 0 else 0,
            "Overall": overall_accuracy
        }

In [11]:

def run_model(filepath, category, chunk_size, method):
    model = ModelHyperparameterTuning(chunk_size=chunk_size, method=method)
    vignettes = model.load_vignettes(filepath, category)
    results = model.evaluate_model(vignettes)
    print("Evaluation Results:", results)
    
run_model("Data_final.csv", "Stress", chunk_size=50, method="rag compress + COT compress")

------------------------------------------------------------------------------
Vignette ID: Vignette 1A FINAL
Category: Stress
Vignette Text:
Mr. WM is a 45-year-old truck driver who was referred by his primary care physician.  Mr. WM mentioned that he was having difficulty on the job since having been in a road crash with his vehicle about one year ago. He spent a week in the hospital recovering from his injuries from the crash, and he stated that he felt lucky to be alive afterwards.
 Since the crash, Mr. WM feels constantly nervous and jumps when he hears loud noises, particularly when walking along the roadway.  When he hears cars braking at high speeds, he feels his arms go forward and his foot involuntarily pushing on an imaginary brake.  Preceding a road trip, his sleep is severely disrupted, due to nightmares about the crash. He paces around the house and feels chest pains along his rib cage where he was pinned down by the steering wheel. He has avoided doing any driving during

------------------------------------------------------------------------------
Vignette ID: Vignette 1D FINAL
Category: Stress
Vignette Text:
Mr. DB is a 47-year-old man who is a self-employed stock market trader.  He was referred by his family physician due to persistent symptoms that are having a serious negative impact on his life that began suddenly following an accident one year ago. Mr. DB is unable to understand why he cannot simply overcome the incident, and was reluctant to see a mental health specialist.
One year ago, Mr. DB was carrying out repairs to a large walk-in cupboard in his house when the ceiling and part of the roof collapsed, leaving him trapped underneath and choking from the clouds of dust that were released. Since that time he has felt nervous most hours of the day, avoids being in public or crowded places, and avoids open cupboards. He is easily startled by unexpected noises. Despite being an avid cook, his fear has generalized to the kitchen. Whenever he ente

------------------------------------------------------------------------------
Vignette ID: Vignette 2B FINAL
Category: Stress
Vignette Text:
FK is a 19-year-old college student. He currently lives in a college dormitory with two roommates. FK has a history of severe physical abuse by his stepfather, beginning when he was 12 years old. He has been hospitalized twice due to physical injuries from these beatings. At age 13, the stepfather broke FK’s arm when he tried to stop his mother from being beaten. Two years ago, the police arrested FK’s stepfather after his mother was brought to a hospital with life-threatening injuries.  The stepfather has been in jail ever since. FK’s current roommates have noticed some peculiar behaviors and referred FK to the school’s counselor.
 During the talk with the counselor, FK was initially evasive about the nature of the visit. When the counselor replied that his roommates had mentioned that he had been beaten, FK started crying and admitted that he a

------------------------------------------------------------------------------
Vignette ID: Vignette 4 FINAL
Category: Stress
Vignette Text:
DD is a 16-year-old boy who was referred to the mental health clinic by his teacher.  She is concerned about his falling grades and his changed personality over the past 18 months, since the death of his brother JD. The older brother volunteered as a soldier at age 17 and was killed a few weeks later in an ongoing war. During the first few months after the loss, DD was often truant from school, and when present, appeared distracted and disinterested in class. Sometimes he was tearful and would not join in any recreation. Before his brother’s death, he had been a star pupil. The teacher said that DD’s reactions were understandable during the first few months following his brother’s death, but she expected him to recover over time and he has not done so. More than 18 months after his brother’s death, DD is still not back to his normal self.  Althoug

------------------------------------------------------------------------------
Vignette ID: Vignette 5B FINAL
Category: Stress
Vignette Text:
AH, a 45-year-old woman, was referred after meeting the school psychologist of her eight year old son one week ago. She describes herself as having suffered from a “nervous breakdown” after her husband was severely injured in an automobile accident three months ago. She was riding with him when they were hit by a much larger vehicle. The husband’s left arm and leg were crushed, and he is currently undergoing reconstructive surgeries. AH witnessed the event but was otherwise unharmed. Since the accident, she is only able to complete household necessities with strenuous effort but often is not able to manage it at all.
 She reports that she is constantly thinking about the accident and her husband’s injuries. Her thoughts on these issues “circle constantly” in her mind. She often cries and is not able to stop. Being with her son does not keep her f

Method:
"rag only" - only rag

"rag compress" - rag compress

"cot only" - only cot

"cot compress" - cot + compress

"RAG + COT" - RAG + COT

"rag compress + COT" - compress RAG + COT

"RAG + COT compress" - RAG + COT compress

"rag compress + COT compress" - compress both RAG & COT